# WatchEvent pipeline test (`07_watch_events`)

This notebook tests `07_watch_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_watch`. It runs the real notebook with `%run` and `test_schema = test_watch`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `07_watch_events` (they create `silver.watch_events` and `quarantine.quarantine_watch_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to WatchEvent: `watch_action` is read, a missing or blank action is kept with a warning, an action other than `started` is kept unchanged with an `UNEXPECTED_VALUE` warning, and the quarantine table keeps `watch_action` as evidence.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_watch"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("watch_events")
TEST_QUARANTINE = T("quarantine_watch_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.watch_events",
    "github_lakehouse.quarantine.quarantine_watch_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 07_watch_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.watch_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_watch_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

| Row | Case | Expected result |
|---|---|---|
| `w1` | clean WatchEvent, `action = started` | Silver, no warning |
| `w2` | `payload = {}` (no action) | Silver, `watch_action` NULL, warning `MISSING: watch_action` |
| `w3` | `action` is blank | Silver, `watch_action` NULL, warning `MISSING: watch_action` |
| `w4` | `action = stopped` | Silver, value kept as `stopped`, warning `UNEXPECTED_VALUE: watch_action` |
| `w5` | `payload` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `w6` | `repo.id` missing | quarantine `MISSING: repo_id`, `watch_action` kept as evidence |
| `w7` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `w8` | `actor.id` missing and `action = unstarred` | Silver, warning `MISSING: actor_id; UNEXPECTED_VALUE: watch_action` (NULL-step warnings first) |
| `w9` | no `payload` at all | Silver, warning `MISSING: watch_action` |
| `x1` | a ForkEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

def ev(event_id, type_="WatchEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, payload=None, drop=()):
    """One GitHub-style event as a JSON string. Defaults = a clean WatchEvent."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": {"action": "started"} if payload is None else payload,
    }
    for key in drop:
        event.pop(key)
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)              # case
    (ev("w1"),                                            F1, D1, I1),   # clean
    (ev("w2", payload={}),                                F1, D1, I1),   # no action -> warn
    (ev("w3", payload={"action": "   "}),                 F1, D1, I1),   # blank action -> NULL + warn
    (ev("w4", payload={"action": "stopped"}),             F1, D1, I1),   # unexpected action -> warn
    (ev("w5", payload="oops"),                            F1, D1, I1),   # payload not an object -> parse error
    (ev("w6", repo={"name": "octo/repo"}),                F1, D1, I1),   # repo_id missing -> quarantine
    (ev("w7", actor={"id": 171, "login": "old"}),         F1, D1, I1),   # duplicate, older
    (ev("w7", actor={"id": 172, "login": "new"}),         F2, D1, I2),   # duplicate, newer -> wins
    (ev("w8", actor={"login": "bob"},
        payload={"action": "unstarred"}),                 F1, D1, I1),   # two warnings
    (ev("w9", drop=["payload"]),                          F1, D1, I1),   # no payload -> warn
    (ev("x1", type_="ForkEvent", payload={}),             F1, D1, I1),   # other event type -> ignored
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 11)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../07_watch_events $test_schema="test_watch" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 1, "null_actor_login": 0,
    "null_repo_name": 0, "null_watch_action": 3, "unexpected_watch_action": 2,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_watch_events")
check("audit event_type", audit["event_type"], "WatchEvent")

# --- audit: 10 WatchEvent rows (x1 is a ForkEvent)
check_audit({
    "bronze_input_rows": 10, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 8,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 7,
    "silver_rows_inserted": 7, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, watch_action before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "watch_action",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      ["w1", "w2", "w3", "w4", "w7", "w8", "w9"])

r = one_silver("w1")
check("w1 action", r.get("watch_action"), "started")
check("w1 ids", (r.get("actor_id"), r.get("repo_id")), (101, 201))
check("w1 warning", r.get("quarantine_warning"), None)

for event_id in ["w2", "w3", "w9"]:
    r = one_silver(event_id)
    check(f"{event_id} action", r.get("watch_action"), None)
    check(f"{event_id} warning", r.get("quarantine_warning"), "MISSING: watch_action")

r = one_silver("w4")
check("w4 action kept unchanged", r.get("watch_action"), "stopped")
check("w4 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: watch_action")

r = one_silver("w7")
check("w7 winner = newer ingestion", (r.get("actor_id"), r.get("_source_file")), (172, F2))
check("w7 warning", r.get("quarantine_warning"), None)

r = one_silver("w8")
check("w8 action kept unchanged", r.get("watch_action"), "unstarred")
check("w8 warnings, NULL step first",
      r.get("quarantine_warning"), "MISSING: actor_id; UNEXPECTED_VALUE: watch_action")

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["JSON_PARSE_ERROR", "MISSING: repo_id"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("w5 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["w5"])
w6 = [r for r in q if r["event_id"] == "w6"]
check("w6 reason", [r["quarantine_reason"] for r in w6], ["MISSING: repo_id"])
check("w6 keeps watch_action as evidence", [r["watch_action"] for r in w6], ["started"])
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../07_watch_events $test_schema="test_watch" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 10, "silver_candidate_rows": 7,
    "silver_rows_inserted": 0, "silver_rows_already_present": 7,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 7)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")